# 🌐 Run Virgo-1.0-Angkor for the Virgo AI website (test server)

Runs your trained **Virgo-1.0-Angkor** on this Colab GPU and gives it a public https address, so virgoai.camksn.com can use it for **chat** and **real-time voice (Virgo Live)**.

1. **Runtime → Change runtime type → T4 GPU**.
2. **Runtime → Run all**. It asks for your Hugging Face token (to download Virgo-1.0-Angkor).
3. Copy the two values it prints into Cloudflare → your Worker → **Settings → Variables and Secrets** as **Secrets**: `VIRGO_API_URL` and `VIRGO_API_KEY`.

⚠️ Test only: it works while this tab is open, and the address changes each run. For a permanent server, use a rented GPU (see the README).

**Your own fixed address (optional):** create a Cloudflare Tunnel (Zero Trust → Networks → Tunnels → Create → Cloudflared, public hostname `virgo.camksn.com` → HTTP `localhost:8000`), then add two secrets here (the 🔑 **Secrets** panel on the left, with notebook access on): `CF_TUNNEL_TOKEN` (the tunnel token) and `VIRGO_API_KEY` (a long password you choose). Set the Worker secrets once to `https://virgo.camksn.com` and the same password; after that, just run this notebook.

In [ ]:
#@title ⚙️ Settings
HF_MODEL = ""  #@param {type:"string"}
#@markdown Leave empty to use `<your Hugging Face name>/Virgo-1.0-Angkor`, or the Virgo you just trained in this Colab.
VIRGO_HOST = "https://virgo.camksn.com"  #@param {type:"string"}
#@markdown Your own address (a named Cloudflare Tunnel), used when the `CF_TUNNEL_TOKEN` secret is set.

In [ ]:
#@title 1️⃣ Set up
import os, subprocess, sys, torch
assert torch.cuda.is_available(), "❌ No GPU: Runtime → Change runtime type → T4 GPU"
subprocess.run("pkill -f 'uvicorn serve.app' ; pkill -f cloudflared", shell=True)
os.chdir("/content")
if not os.path.isdir("virgo-models/.git"):
    subprocess.run(["git", "clone", "-q", "https://github.com/nangmrr752/virgo-models"], check=True)
os.chdir("/content/virgo-models")
subprocess.run(["git", "pull", "-q"])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-serve.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])  # conflicts with newer peft
from huggingface_hub import login, HfApi, snapshot_download
from getpass import getpass
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = None
login(token=token or getpass("Hugging Face token (hf_...): ").strip())
ADAPTER = "chat/out/virgo-1.0-chat-lora"
if not os.path.isdir(ADAPTER):
    repo = HF_MODEL or f"{HfApi().whoami()['name']}/Virgo-1.0-Angkor"
    print("⬇️ Downloading", repo)
    snapshot_download(repo, local_dir=ADAPTER)
print("✅ Ready: Virgo-1.0-Angkor is in", ADAPTER)

In [ ]:
#@title 2️⃣ Start the Virgo server and open a public address
import os, subprocess, time, re, secrets, json, urllib.request
os.chdir("/content/virgo-models")
def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None
SAVED_KEY = secret("VIRGO_API_KEY")
KEY = SAVED_KEY or secrets.token_urlsafe(32)  # a saved key stays the same every run
TUNNEL = secret("CF_TUNNEL_TOKEN")           # your own tunnel: the address never changes
env = {**os.environ, "VIRGO_API_KEY": KEY, "VIRGO_PRELOAD": "chat,speech"}
subprocess.Popen("uvicorn serve.app:app --host 0.0.0.0 --port 8000 > /content/server.log 2>&1", shell=True, env=env)
print("⏳ Loading Virgo-1.0-Angkor (1–3 minutes)...")
for _ in range(180):
    time.sleep(2)
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5); break
    except Exception:
        pass
else:
    print("".join(open("/content/server.log").readlines()[-40:])); raise SystemExit("❌ The server didn't start: send the lines above to Claude.")
req = urllib.request.Request("http://localhost:8000/v1/chat", method="POST",
    data=json.dumps({"messages": [{"role": "user", "content": "Who are you?"}], "max_tokens": 60}).encode(),
    headers={"content-type": "application/json", "authorization": f"Bearer {KEY}"})
try:
    print("🤖", json.loads(urllib.request.urlopen(req, timeout=600).read())["reply"])
except Exception as err:
    print("".join(open("/content/server.log").readlines()[-40:])); raise SystemExit(f"❌ {err}: send the lines above to Claude.")
if not os.path.exists("/content/cloudflared"):
    subprocess.run("wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared", shell=True)
subprocess.run("pkill -f 'cloudflared tunnel'", shell=True)
log = "/content/tunnel.log"
if TUNNEL:
    # The token goes in the environment, never on the command line or in the output.
    subprocess.Popen(f"/content/cloudflared tunnel --no-autoupdate run > {log} 2>&1", shell=True, env={**os.environ, "TUNNEL_TOKEN": TUNNEL})
    for _ in range(60):
        time.sleep(2)
        if "Registered tunnel connection" in open(log).read(): break
    else:
        print(open(log).read()[-2000:]); raise SystemExit("❌ Your tunnel didn't connect: check the CF_TUNNEL_TOKEN secret (lines above).")
    url = VIRGO_HOST.rstrip("/")
else:
    subprocess.Popen(f"/content/cloudflared tunnel --url http://localhost:8000 > {log} 2>&1", shell=True)
    url = None
    for _ in range(60):
        time.sleep(2)
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(log).read())
        if m: url = m.group(0); break
    if not url:
        print(open(log).read()[-2000:]); raise SystemExit("❌ No public address: send the lines above to Claude.")
print("\n✅ Virgo server is live (chat + real-time voice) at", url)
if TUNNEL and SAVED_KEY:
    print("Your own address and saved key: nothing to change in Cloudflare (VIRGO_API_URL =", url + ", VIRGO_API_KEY = your saved key).")
else:
    print("Add these as Cloudflare Worker SECRETS:")
    print("VIRGO_API_URL =", url)
    print("VIRGO_API_KEY =", "your saved VIRGO_API_KEY secret" if SAVED_KEY else KEY)
print("\nKeep this tab open. If something goes wrong: !tail -40 /content/server.log")
